In [1]:
!git clone https://github.com/Miki004/VISTA.git
%cd VISTA

fatal: destination path 'VISTA' already exists and is not an empty directory.
/workspace/VISTA


In [2]:
import sys
print("Python kernel:", sys.executable)
# deve essere /venv/main/bin/python

import torch
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

Python kernel: /venv/main/bin/python
torch version: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition


In [4]:
"""Esegue la pipeline QwenYolo su una cartella di video e salva i risultati."""
from __future__ import annotations

import csv
import argparse
import ultralytics
from pathlib import Path
from collections import defaultdict

from ultralytics import YOLO

from vista.pipeline.qwen_yolo import QwenYoloPipeline
from vista.qwen import get_model   # <-- il modulo Qwen del repo della challenge


In [ ]:
import sys
sys.path.insert(0, '/workspace/VISTA')

import csv
from pathlib import Path
from collections import defaultdict

from ultralytics import YOLO
from vista.pipeline.qwen_yolo import QwenYoloPipeline
from vista.qwen import get_model

# === CONFIG ===
VIDEOS_DIR = Path("/workspace/VISTA/workspace/vista_videos")
OUT_DIR = Path("/workspace/VISTA/workspace/vista_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = {
"device": "cuda:0",

"yolo": {
"model": "yolo12m.pt",
"iou_match_threshold": 0.3,
},

"qwen": {

"model_id": "Qwen/Qwen3-VL-8B-Instruct",
"every_n_frames": 10,
"max_new_tokens": 4096,
"image_target_size": 1024,
"system_prompt": 
    """You are an operator supervising a drone operation over an accident scene. Your task is to detect and label all relevant objects in the images. Focus on the following:

    1. Vehicles:
    - Identify and classify all vehicles, including cars, trucks, motorcycles, bicycles only if they are involved in the accident.
    - Distinguish between:
        * Vehicles involved in the accident
        * Emergency or helping vehicles

    2. People:
    - Detect all people present in the scene.
    - Describe their actions and status, including but not limited to: injured, hurt, standing, sitting, walking, running, helping others, calling for help, etc.
    - Include this information in the label.

    Output format:
    - Return a valid JSON array with bounding boxes for all detected elements in the form:
    `[{"bbox_2d": [xmin, ymin, xmax, ymax], "label": "detailed description"}, ...]`
    - Example valid response:
    `[{"bbox_2d": [10, 30, 20, 60], "label": "car involved in accident"}, {"bbox_2d": [40, 15, 52, 27], "label": "person injured, sitting"}]`
    - Ensure each object is labeled with a precise description reflecting its type and status.""",

    "user_prompt":"Detect and describe all accident-relevant objects in this frame.",
    },
}


# === LOAD MODELS (una volta sola) ===
print("Carico YOLO...")
yolo = YOLO(cfg["yolo"]["model"])

print("Carico Qwen (può richiedere alcuni minuti la prima volta)...")
qwen = get_model(cfg)

pipeline = QwenYoloPipeline(yolo_model=yolo, qwen_model=qwen, caption_stride=cfg["qwen"]["every_n_frames"])
print("✓ Pipeline pronta")

In [ ]:
# === PROCESSA I VIDEO ===
video_paths = sorted(
    p for p in VIDEOS_DIR.iterdir()
    if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"}
)
print(f"Trovati {len(video_paths)} video")

csv_path = OUT_DIR / "submission.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["video_id", "track_id", "frame_start", "frame_end", "caption"])

    for video_path in video_paths:
        video_id = video_path.stem
        print(f"\n=== {video_id} ===")

        tracks = defaultdict(lambda: {"frame_start": None, "frame_end": None, "caption": None})

        for result in pipeline.process_video(str(video_path)):
            if result.frame_idx % 30 == 0:
                print(f"  frame {result.frame_idx}: {len(result.detections)} detections")
            for det in result.detections:
                if det.track_id is None:
                    continue
                rec = tracks[det.track_id]
                if rec["frame_start"] is None:
                    rec["frame_start"] = result.frame_idx
                rec["frame_end"] = result.frame_idx
                if det.caption:
                    rec["caption"] = det.caption

        for tid, rec in tracks.items():
            writer.writerow([video_id, tid, rec["frame_start"], rec["frame_end"], rec["caption"] or ""])

print(f"\n✓ Submission salvata in {csv_path}")

In [ ]:
print("Device del modello:", next(qwen.model.parameters()).device)
print("Pesi totali:", sum(p.numel() for p in qwen.model.parameters()) / 1e9, "B")

# vedi se ci sono pesi su CPU
cpu_params = [n for n, p in qwen.model.named_parameters() if p.device.type == 'cpu']
print(f"Parametri su CPU: {len(cpu_params)}")
if cpu_params:
    print("Esempi:", cpu_params[:5])

Genera dei video annotati(boundingboxes + captioning)

In [ ]:
!{sys.executable} -m vista.annotate_video --video input.mp4 --out annotated.mp4 --config config/qwenyolo/mycfg.yaml --yolo-weights yolo12m.pt --start-frame 6000 --end-frame 7000   

Esegue una valutazione del detector(YOLO) su VistaCrash calcolando la mAP@

In [ ]:
!{sys.executable} -m vista.eval_detection --data data/VistaCrash/data.yaml \
        --split val --weights yolo12m.pt --conf 0.25

Genera un submission.csv usando la qwen_yolo baseline.
Esegue una VistaPipeline per ogni video all'interno di una cartella, aggrega per-track
results, e scrive il CSV successivamente processato da ``vista.eval_era``:
    video_id, track_id, frame_start, frame_end, caption

In [ ]:
!{sys.executable} -m vista.run_submission --videos data/ERA/TrafficCollision \
        --out workspace/vista_output/submission.csv  --skip-empty \
        --config config/qwenyolo/cfg7.yaml --yolo-weights yolo12m.pt

In [ ]:
!{sys.executable} -m vista.eval_era --references CapERA_DATASET_train.json \
        --submission workspace/vista_output/submission.csv

Esegue MyPipeline la quale passa al VLM i crops di YOLO

In [ ]:
from vista.pipeline.mypipeline import build_mypipeline_from_config

pipeline = build_mypipeline_from_config(
    config_path="config/qwenyolo/mycfg.yaml",
    yolo_weights="yolo12m.pt",
)

from vista.run_submission import generate_submission
generate_submission(pipeline, "VistaVideos", "workspace/vista_output/MyPipelineSubmission.csv")

In [ ]:
COCO_CATEGORY_MAP = {
    "car": "car", "truck": "car", "bus": "car", "motorcycle": "car",
    "person": "person",
}
YOLOE_CATEGORY_MAP = {
    "crashed car": "car", "car": "car", "person": "person",
}

In [ ]:
from ultralytics import YOLO, YOLOE
from vista.qwen import get_model
import yaml

cfg  = yaml.safe_load(open("mycfg.yaml"))  #path of the mycfg 
qcfg = cfg.get("qwen", {})
captioner = QwenCropCaptioner(get_model(cfg), system_prompt=qcfg.get("system_prompt"))  # uno solo

def make_pipe(detector, category_map):
    return MyPipeline(detector, captioner, category_map=category_map,
                      caption_stride=qcfg.get("every_n_frames", 30),
                      crop_frac=0.1, min_crop_size=256)

coco_pipe  = make_pipe(YOLO("yolo12m.pt"),                                   COCO_CATEGORY_MAP)
yoloe_pipe = make_pipe(YOLOE("runs/detect/vistacrash_lp/weights/best.pt"),   YOLOE_CATEGORY_MAP)

In [ ]:
import cv2
from pathlib import Path
from PIL import Image

def run_on_video(pipeline, video_in, video_out):
    cap = cv2.VideoCapture(video_in)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Path(video_out).parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(video_out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    pipeline.reset()
    all_tracks, captioned = set(), set()
    idx = 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        frame = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        res = pipeline.forward(frame, idx)
        for det in res.detections:
            all_tracks.add(det.track_id)
            if det.caption:
                captioned.add(det.track_id)
            x1, y1, x2, y2 = map(int, det.bbox)
            color = (0, 200, 0) if det.caption else (0, 0, 255)   # verde=captionato, rosso=no
            cv2.rectangle(bgr, (x1, y1), (x2, y2), color, 2)
            txt = f"{det.track_id} {det.category}: {det.caption or '...'}"
            cv2.putText(bgr, txt, (x1, max(12, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
        writer.write(bgr)
        idx += 1
    cap.release(); writer.release()
    cov = len(captioned) / max(1, len(all_tracks))
    print(f"{Path(video_out).name}: {idx} frame | {len(all_tracks)} track | "
          f"coverage {cov:.1%} ({len(captioned)}/{len(all_tracks)})")
    return cov

In [ ]:
videos = ["data/sequences/bike_3.mp4", "data/sequences/era_collision.mp4"]  # i tuoi
for v in videos:
    name = Path(v).stem
    run_on_video(coco_pipe,  v, f"out/compare/{name}_coco.mp4")
    run_on_video(yoloe_pipe, v, f"out/compare/{name}_yoloe.mp4")